## Statistical Hypothesis Testing

This section evaluates statistically significant relationships
in the Online Retail dataset to support evidence-based
business decisions.

The analysis will include:
- Research questions and hypotheses
- Selection of appropriate statistical tests
- Significance level (α = 0.05)
- Test results and p-values
- Business interpretation and recommendations

### Hypothesis 1: Customer Spending and Repeat Purchases

**Research Question:** Do repeat customers spend more than
one-time customers?

**H₀ (Null Hypothesis):**
The mean total spending of repeat customers is equal to
the mean total spending of one-time customers.

**H₁ (Alternative Hypothesis):**
The mean total spending differs between repeat customers
and one-time customers.

**Significance Level:** α = 0.05

**Unit of Analysis:** Individual customers.

**Planned Test:** Welch's independent samples t-test,
subject to data validation.

**Business Relevance:**
Understanding spending differences between repeat and
one-time customers can help inform customer retention
and marketing strategies.

In [1]:
# STEP 1: Load customer intelligence data

import pandas as pd
import numpy as np
from pathlib import Path

# Locate the project directory
project_root = Path.cwd().resolve()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

# Load customer-level data
customer_path = (
    project_root
    / "data"
    / "tableau"
    / "retail_customer_intelligence.csv"
)

customer_df = pd.read_csv(customer_path)

print("Customer dataset shape:", customer_df.shape)

# Inspect available columns
print("\nColumn names:")
print(customer_df.columns.tolist())

# Preview first five rows
display(customer_df.head())

Customer dataset shape: (4337, 10)

Column names:
['CustomerID', 'TotalSpending', 'TotalOrders', 'TotalUnits', 'FirstPurchase', 'LastPurchase', 'PurchaseFrequency', 'AverageOrderValue', 'CustomerType', 'RecencyDays']


,CustomerID,TotalSpending,TotalOrders,TotalUnits,FirstPurchase,LastPurchase,PurchaseFrequency,AverageOrderValue,CustomerType,RecencyDays
0,12346,77183.60,1,74215,2011-01-18 10:01:00,2011-01-18 10:01:00,1,77183.600000,One-Time Customer,326
1,12347,4310.00,7,2458,2010-12-07 14:57:00,2011-12-07 15:52:00,7,615.714286,Repeat Customer,3
2,12348,1797.24,4,2341,2010-12-16 19:09:00,2011-09-25 13:13:00,4,449.310000,Repeat Customer,76
3,12349,1757.55,1,631,2011-11-21 09:51:00,2011-11-21 09:51:00,1,1757.550000,One-Time Customer,19
4,12350,334.40,1,197,2011-02-02 16:01:00,2011-02-02 16:01:00,1,334.400000,One-Time Customer,311


In [2]:
# STEP 2: Validate customer data for hypothesis testing

# Check missing values in required columns
print("Missing values:")
print(
    customer_df[
        ["CustomerID", "TotalSpending", "TotalOrders", "CustomerType"]
    ].isnull().sum()
)

# Check duplicate customer IDs
print("\nDuplicate customer IDs:")
print(customer_df["CustomerID"].duplicated().sum())

# Check invalid spending values
print("\nCustomers with zero or negative spending:")
print((customer_df["TotalSpending"] <= 0).sum())

# Check customer group sizes
print("\nCustomer type distribution:")
print(customer_df["CustomerType"].value_counts())

# Verify customer classification
classification_errors = (
    (
        (customer_df["CustomerType"] == "One-Time Customer")
        & (customer_df["TotalOrders"] != 1)
    )
    |
    (
        (customer_df["CustomerType"] == "Repeat Customer")
        & (customer_df["TotalOrders"] <= 1)
    )
)

print("\nCustomer classification inconsistencies:")
print(classification_errors.sum())

Missing values:
CustomerID       0
TotalSpending    0
TotalOrders      0
CustomerType     0
dtype: int64

Duplicate customer IDs:
0

Customers with zero or negative spending:
0

Customer type distribution:
CustomerType
Repeat Customer      2844
One-Time Customer    1493
Name: count, dtype: int64

Customer classification inconsistencies:
0


In [3]:
# STEP 3: Prepare customer groups for hypothesis testing

# Extract total spending for repeat customers
repeat_spending = customer_df.loc[
    customer_df["CustomerType"] == "Repeat Customer",
    "TotalSpending"
]

# Extract total spending for one-time customers
one_time_spending = customer_df.loc[
    customer_df["CustomerType"] == "One-Time Customer",
    "TotalSpending"
]

# Display descriptive statistics for each group
print("REPEAT CUSTOMERS")
print(repeat_spending.describe())

print("\nONE-TIME CUSTOMERS")
print(one_time_spending.describe())

# Compare average spending
print("\nAVERAGE CUSTOMER SPENDING")
print(f"Repeat customers: £{repeat_spending.mean():,.2f}")
print(f"One-time customers: £{one_time_spending.mean():,.2f}")

# Compare median spending
print("\nMEDIAN CUSTOMER SPENDING")
print(f"Repeat customers: £{repeat_spending.median():,.2f}")
print(f"One-time customers: £{one_time_spending.median():,.2f}")

REPEAT CUSTOMERS
count      2844.000000
mean       2908.837076
std       10901.360364
min           6.900000
25%         615.777500
50%        1155.535000
75%        2433.850000
max      280206.020000
Name: TotalSpending, dtype: float64

ONE-TIME CUSTOMERS
count     1493.000000
mean       411.245520
std       2034.246264
min          3.750000
25%        152.650000
50%        255.910000
75%        406.090000
max      77183.600000
Name: TotalSpending, dtype: float64

AVERAGE CUSTOMER SPENDING
Repeat customers: £2,908.84
One-time customers: £411.25

MEDIAN CUSTOMER SPENDING
Repeat customers: £1,155.53
One-time customers: £255.91


In [4]:
# STEP 4: Examine spending skewness and potential outliers

import pandas as pd

for group_name, spending in [
    ("Repeat Customers", repeat_spending),
    ("One-Time Customers", one_time_spending)
]:
    q1 = spending.quantile(0.25)
    q3 = spending.quantile(0.75)
    iqr = q3 - q1

    upper_bound = q3 + 1.5 * iqr
    outlier_count = (spending > upper_bound).sum()

    print(f"\n{group_name.upper()}")
    print(f"Skewness: {spending.skew():.2f}")
    print(f"IQR: £{iqr:,.2f}")
    print(f"Upper outlier boundary: £{upper_bound:,.2f}")
    print(f"Potential high-spending outliers: {outlier_count:,}")
    print(f"Outlier percentage: {outlier_count / len(spending) * 100:.2f}%")


REPEAT CUSTOMERS
Skewness: 16.17
IQR: £1,818.07
Upper outlier boundary: £5,160.96
Potential high-spending outliers: 256
Outlier percentage: 9.00%

ONE-TIME CUSTOMERS
Skewness: 36.13
IQR: £253.44
Upper outlier boundary: £786.25
Potential high-spending outliers: 105
Outlier percentage: 7.03%


In [5]:
import scipy
from scipy import stats

print("SciPy version:", scipy.__version__)
print("SciPy imported successfully!")

SciPy version: 1.13.1
SciPy imported successfully!


In [6]:
# STEP 5: Prepare customer spending groups for hypothesis testing

repeat_spending = customer_df.loc[
    customer_df["CustomerType"] == "Repeat Customer",
    "TotalSpending"
].dropna()

one_time_spending = customer_df.loc[
    customer_df["CustomerType"] == "One-Time Customer",
    "TotalSpending"
].dropna()

print("CUSTOMER GROUPS FOR HYPOTHESIS TESTING")
print(f"Repeat customers: {len(repeat_spending):,}")
print(f"One-time customers: {len(one_time_spending):,}")

print("\nAVERAGE SPENDING")
print(f"Repeat customers: £{repeat_spending.mean():,.2f}")
print(f"One-time customers: £{one_time_spending.mean():,.2f}")

CUSTOMER GROUPS FOR HYPOTHESIS TESTING
Repeat customers: 2,844
One-time customers: 1,493

AVERAGE SPENDING
Repeat customers: £2,908.84
One-time customers: £411.25


In [7]:
# STEP 6: Welch's Independent-Samples T-Test

# Perform a two-sided Welch's t-test
t_statistic, p_value = stats.ttest_ind(
    repeat_spending,
    one_time_spending,
    equal_var=False
)

# Set significance level
alpha = 0.05

print("WELCH'S INDEPENDENT-SAMPLES T-TEST")
print("-" * 45)

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if p_value < alpha:
    print("Reject the null hypothesis (H0).")
    print("There is a statistically significant difference")
    print("in mean spending between the two customer groups.")
else:
    print("Fail to reject the null hypothesis (H0).")
    print("There is insufficient evidence of a difference")
    print("in mean spending between the two customer groups.")

WELCH'S INDEPENDENT-SAMPLES T-TEST
---------------------------------------------
T-statistic: 11.8320
P-value: 1.20273e-31
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference
in mean spending between the two customer groups.


In [8]:
# STEP 7: Mann-Whitney U Test

# Compare repeat and one-time customer spending
u_statistic, mw_p_value = stats.mannwhitneyu(
    repeat_spending,
    one_time_spending,
    alternative="two-sided"
)

alpha = 0.05

print("MANN-WHITNEY U TEST")
print("-" * 45)

print(f"U-statistic: {u_statistic:,.2f}")
print(f"P-value: {mw_p_value:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if mw_p_value < alpha:
    print("Reject the null hypothesis (H0).")
    print("There is a statistically significant difference")
    print("in the spending distributions of the two groups.")
else:
    print("Fail to reject the null hypothesis (H0).")
    print("There is insufficient evidence of a difference")
    print("in the spending distributions of the two groups.")

MANN-WHITNEY U TEST
---------------------------------------------
U-statistic: 3,787,067.00
P-value: 0
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference
in the spending distributions of the two groups.


### Step 8: Hypothesis Testing Results and Business Interpretation

**Research Question:** Do repeat customers spend more than one-time customers?

**Statistical Results:**

- Welch's independent-samples t-test: t = 11.8320, p ≈ 1.20 × 10⁻³¹.
- Mann–Whitney U test: U = 3,787,067, p extremely small (reported as 0 due to numerical precision).
- Significance level: α = 0.05.

**Descriptive Findings:**

- Repeat customers (n = 2,844): Mean spending = £2,908.84; median = £1,155.53.
- One-time customers (n = 1,493): Mean spending = £411.25; median = £255.91.
- Difference in mean spending: approximately £2,497.59.

**Conclusion:**

Both statistical tests reject their respective null hypotheses. Repeat customers demonstrate significantly higher average spending, and the spending distributions differ significantly between the two groups.

**Business Implications:**

The retailer should consider targeted retention campaigns, personalised offers, and loyalty initiatives to encourage repeat purchases and strengthen customer relationships.

**Limitations:**

Customer spending is highly right-skewed and contains substantial outliers. Welch's t-test compares means, while the Mann–Whitney U test compares distributions. The findings demonstrate statistical associations rather than causal effects. Furthermore, repeat customers have more purchasing opportunities by definition, which contributes to their higher cumulative spending.